# MedRAX Premium — Kaggle Benchmark

**Goal:** Run ChestAgentBench (2,500 questions) with conflict resolution and measure:
- Conflict Occurrence Rate (how many questions trigger conflicts)
- Conflict Detection Ratio (conflicts per question)
- Conflict Resolve Ratio (% resolved without human deferral)

**Output:** Table 1 comparison adding **MedRAX_Premium** column alongside LLaVA-Med, CheXagent, Llama-3.2-90B, GPT-4o, MedRAX

**Requirements:** Kaggle GPU T4 x2, Dataset `medrax-premium` attached


## Cell 1: Setup

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1: COMPLETE SETUP — install, patch, download, verify
# ══════════════════════════════════════════════════════════════
import os, sys, shutil, json, time, glob

# ─────────────────────────────────────────────────────────────
# STEP 1: Copy project to writable location (CODE ONLY — symlink figures)
# ─────────────────────────────────────────────────────────────
SRC = "/kaggle/input/datasets/mohammadninadmahmud/medrax-premium-new/Medrax_premium"
ROOT = "/kaggle/working/medrax_premium"

# Model cache goes to /tmp (more space than /kaggle/working which is ~20 GB)
MODEL_DIR = "/tmp/hf_cache"

if not os.path.exists(os.path.join(SRC, "medrax", "agent", "agent.py")):
    for name in os.listdir("/kaggle/input"):
        for root_d, dirs, files in os.walk(os.path.join("/kaggle/input", name)):
            if os.path.exists(os.path.join(root_d, "medrax", "agent", "agent.py")):
                SRC = root_d
                break

assert os.path.exists(os.path.join(SRC, "medrax", "agent", "agent.py")), \
    f"❌ Project not found! Check dataset upload."
print(f"✅ Found project at: {SRC}")

if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

# Copy code only — skip large figures/ dir to save ~3 GB on /kaggle/working
def _ignore_figures(src, names):
    if os.path.basename(src) == "chestagentbench" and "figures" in names:
        return {"figures"}
    return set()

shutil.copytree(SRC, ROOT, ignore=_ignore_figures)
os.makedirs(MODEL_DIR, exist_ok=True)

# Symlink figures from read-only input (avoids copying thousands of images)
fig_src = os.path.join(SRC, "chestagentbench", "figures")
fig_dst = os.path.join(ROOT, "chestagentbench", "figures")
if os.path.exists(fig_src) and not os.path.exists(fig_dst):
    os.symlink(fig_src, fig_dst)
    print(f"✅ Symlinked figures → {fig_src}")
print(f"✅ Copied code to: {ROOT}")

# ─────────────────────────────────────────────────────────────
# STEP 2: Apply transformers monkey-patch
# ─────────────────────────────────────────────────────────────
init_file = os.path.join(ROOT, "medrax", "llava", "model", "__init__.py")
with open(init_file, "w") as f:
    f.write(
        "import transformers.utils\n"
        "if not hasattr(transformers.utils, 'add_model_info_to_auto_map'):\n"
        "    transformers.utils.add_model_info_to_auto_map = lambda *args, **kwargs: None\n\n"
        "from .language_model.llava_mistral import LlavaMistralForCausalLM, LlavaMistralConfig\n"
    )
print("✅ Applied transformers monkey-patch")

# ─────────────────────────────────────────────────────────────
# STEP 3: Install compatible packages (PINNED versions)
# ─────────────────────────────────────────────────────────────
print("\n📦 Installing packages (this takes 2-3 min)...")
t0 = time.time()

os.system(
    "pip install -q --force-reinstall "
    "'langchain>=0.3.27,<0.4' "
    "'langchain-core>=0.3.72,<1.0.0' "
    "'langchain-text-splitters>=0.3.9,<1.0.0' "
    "'langchain-openai>=0.3,<0.4' "
    "'langchain-community>=0.3,<0.4' "
    "'langgraph>=0.2.60,<1.0.0' "
    "2>&1 | tail -5"
)
print("  langchain ecosystem: ✅")

os.system(
    "pip install -q openai torchxrayvision pydicom einops accelerate peft "
    "bitsandbytes timm diffusers python-dotenv tenacity tiktoken shortuuid "
    "gradio huggingface_hub 'numpy>=2.0,<2.1' 2>&1 | tail -3"
)
print(f"  other packages: ✅")
print(f"  install time: {time.time()-t0:.0f}s")

# ─────────────────────────────────────────────────────────────
# STEP 4: Set API keys + path
# ─────────────────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["OPENAI_BASE_URL"] = "https://models.inference.ai.azure.com"
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
os.environ["HUGGING_FACE_HUB_TOKEN"] = "YOUR_HF_TOKEN"
sys.path.insert(0, ROOT)
print("✅ API keys set (OpenAI + HuggingFace), path configured")

# ─────────────────────────────────────────────────────────────
# STEP 5: Verify package versions
# ─────────────────────────────────────────────────────────────
print("\n📋 Package versions:")
import importlib.metadata
import langchain_core, langgraph, torch, numpy as np, transformers
print(f"  langchain-core:  {langchain_core.__version__}")
try:
    print(f"  langgraph:       {langgraph.__version__}")
except AttributeError:
    print(f"  langgraph:       {importlib.metadata.version('langgraph')}")
print(f"  transformers:    {transformers.__version__}")
print(f"  torch:           {torch.__version__}")
print(f"  numpy:           {np.__version__}")
print(f"  GPU:             {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY'}")

# ─────────────────────────────────────────────────────────────
# STEP 6: Verify core imports
# ─────────────────────────────────────────────────────────────
print("\n🔗 Core imports:")
try:
    from medrax.agent import Agent, ConflictDetector, ConflictResolver
    print("  ✅ medrax.agent (Agent, ConflictDetector, ConflictResolver)")
except Exception as e:
    print(f"  ❌ medrax.agent FAILED: {e}")

try:
    from medrax.tools import (
        ChestXRayClassifierTool, ChestXRaySegmentationTool,
        ImageVisualizerTool, DicomProcessorTool,
        XRayVQATool, ChestXRayReportGeneratorTool,
        LlavaMedTool, XRayPhraseGroundingTool,
        ChestXRayGeneratorTool,
    )
    print("  ✅ medrax.tools (all 9 tool classes imported)")
except Exception as e:
    print(f"  ❌ medrax.tools FAILED: {e}")

try:
    from medrax.utils import load_prompts_from_file
    print("  ✅ medrax.utils")
except Exception as e:
    print(f"  ❌ medrax.utils FAILED: {e}")

try:
    from langgraph.checkpoint.memory import MemorySaver
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage
    print("  ✅ langchain/langgraph (ChatOpenAI, MemorySaver, HumanMessage)")
except Exception as e:
    print(f"  ❌ langchain FAILED: {e}")

# ─────────────────────────────────────────────────────────────
# STEP 7: Pre-download CORE models (small + CheXagent)
# ─────────────────────────────────────────────────────────────
# Pre-download only core models here. Large models (LLaVA-Med ~15GB,
# MAIRA-2 ~14GB) are downloaded lazily in Cell 2 when tools are loaded.
# This avoids filling /kaggle/working — MODEL_DIR is on /tmp.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n📦 Downloading core models to {MODEL_DIR} (device={DEVICE})...")
models_ok = {}

# Model 1: torchxrayvision DenseNet (classifier) — ~30 MB
print("  [1/5] DenseNet classifier (torchxrayvision)...", end=" ", flush=True)
try:
    import torchxrayvision as xrv
    m = xrv.models.DenseNet(weights="densenet121-res224-all")
    m.to(DEVICE).eval()
    models_ok["classifier"] = True
    del m; torch.cuda.empty_cache()
    print("✅")
except Exception as e:
    models_ok["classifier"] = False
    print(f"❌ {str(e)[:100]}")

# Model 2: torchxrayvision PSPNet (segmentation) — ~50 MB
print("  [2/5] PSPNet segmentation (torchxrayvision)...", end=" ", flush=True)
try:
    m = xrv.baseline_models.chestx_det.PSPNet()
    m.to(DEVICE).eval()
    models_ok["segmentation"] = True
    del m; torch.cuda.empty_cache()
    print("✅")
except Exception as e:
    models_ok["segmentation"] = False
    print(f"❌ {str(e)[:100]}")

# Model 3: CheXagent VQA — ~6 GB VRAM, ~12 GB disk
print("  [3/5] CheXagent-2-3b VQA (HuggingFace ~12GB disk)...", end=" ", flush=True)
t0 = time.time()
try:
    from huggingface_hub import snapshot_download
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from pathlib import Path

    chex_local = snapshot_download("StanfordAIMI/CheXagent-2-3b", cache_dir=MODEL_DIR)

    # Patch the strict version assert in modeling_chexagent.py
    for py_file in glob.glob(str(Path(chex_local) / "*.py")):
        try:
            with open(py_file, "r", encoding="utf-8") as fh:
                content = fh.read()
            if 'assert transformers.__version__' in content:
                content = content.replace(
                    'assert transformers.__version__ == "4.40.0"',
                    '# assert transformers.__version__ == "4.40.0"  # Patched',
                )
                with open(py_file, "w", encoding="utf-8") as fh:
                    fh.write(content)
                print("(patched) ", end="", flush=True)
        except Exception:
            pass

    orig_ver = transformers.__version__
    transformers.__version__ = "4.40.0"
    tok = AutoTokenizer.from_pretrained(chex_local, trust_remote_code=True)
    # Test-load only — will be reloaded in Cell 2 with 4-bit quantization
    mdl = AutoModelForCausalLM.from_pretrained(chex_local, trust_remote_code=True, device_map="cpu")
    transformers.__version__ = orig_ver
    models_ok["chexagent_vqa"] = True
    del tok, mdl; torch.cuda.empty_cache()
    print(f"✅ ({time.time()-t0:.0f}s)")
except Exception as e:
    try: transformers.__version__ = orig_ver
    except: pass
    models_ok["chexagent_vqa"] = False
    print(f"❌ {str(e)[:120]}")

# Model 4: Report gen — findings (~245 MB)
print("  [4/5] Report gen - findings (HuggingFace)...", end=" ", flush=True)
try:
    from transformers import VisionEncoderDecoderModel, BertTokenizer, ViTImageProcessor
    VisionEncoderDecoderModel.from_pretrained("IAMJB/chexpert-mimic-cxr-findings-baseline", cache_dir=MODEL_DIR)
    BertTokenizer.from_pretrained("IAMJB/chexpert-mimic-cxr-findings-baseline", cache_dir=MODEL_DIR)
    ViTImageProcessor.from_pretrained("IAMJB/chexpert-mimic-cxr-findings-baseline", cache_dir=MODEL_DIR)
    models_ok["report_findings"] = True
    torch.cuda.empty_cache()
    print("✅")
except Exception as e:
    models_ok["report_findings"] = False
    print(f"❌ {str(e)[:80]}")

# Model 5: Report gen — impression (~271 MB)
print("  [5/5] Report gen - impression (HuggingFace)...", end=" ", flush=True)
try:
    VisionEncoderDecoderModel.from_pretrained("IAMJB/chexpert-mimic-cxr-impression-baseline", cache_dir=MODEL_DIR)
    BertTokenizer.from_pretrained("IAMJB/chexpert-mimic-cxr-impression-baseline", cache_dir=MODEL_DIR)
    ViTImageProcessor.from_pretrained("IAMJB/chexpert-mimic-cxr-impression-baseline", cache_dir=MODEL_DIR)
    models_ok["report_impression"] = True
    torch.cuda.empty_cache()
    print("✅")
except Exception as e:
    models_ok["report_impression"] = False
    print(f"❌ {str(e)[:80]}")

# LLaVA-Med (~15 GB disk) and MAIRA-2 (~14 GB disk) are downloaded
# lazily in Cell 2 when their tools are instantiated.
# This avoids disk-space pressure in Cell 1.
print("  [info] LLaVA-Med + MAIRA-2 will download in Cell 2 (lazy load)")
print("  [skip] RoentGen — no public weights (not needed for benchmark)")

# ─────────────────────────────────────────────────────────────
# STEP 8: Verify dataset — figures + metadata
# ─────────────────────────────────────────────────────────────
print("\n📂 Dataset verification:")
fig_dir = os.path.join(ROOT, "chestagentbench", "figures")
meta_path = os.path.join(ROOT, "chestagentbench", "metadata.jsonl")

if os.path.exists(fig_dir):
    cases = [d for d in os.listdir(fig_dir) if os.path.isdir(os.path.join(fig_dir, d))]
    total_imgs = sum(len([f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))])
                     for _, _, files in os.walk(fig_dir))
    print(f"  Case folders:  {len(cases):,}")
    print(f"  Image files:   {total_imgs:,}")
    if len(cases) >= 900:
        print("  ✅ Figures OK")
    else:
        print("  ⚠ Figures incomplete!")
else:
    total_imgs = 0
    print("  ❌ figures/ directory NOT FOUND")

questions = []
missing = 0
if os.path.exists(meta_path):
    with open(meta_path) as f:
        questions = [json.loads(line.strip()) for line in f if line.strip()]
    print(f"  Questions:     {len(questions):,}  (expect 2,500)")

    total_refs = 0
    for q in questions:
        for img in q.get("images", []):
            total_refs += 1
            if not os.path.exists(os.path.join(ROOT, "chestagentbench", img)):
                missing += 1
    print(f"  Image refs:    {total_refs:,} total, {missing} missing")
    if missing == 0:
        print("  ✅ All image references resolve")
    else:
        print(f"  ⚠ {missing} images missing!")
else:
    print("  ❌ metadata.jsonl NOT FOUND")

prompts_file = os.path.join(ROOT, "medrax", "docs", "system_prompts.txt")
print(f"  System prompts: {'✅ exists' if os.path.exists(prompts_file) else '❌ MISSING'}")

# ─────────────────────────────────────────────────────────────
# STEP 9: Test GPT-4o API connection
# ─────────────────────────────────────────────────────────────
print("\n🌐 Testing GPT-4o API...")
try:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage
    test_model = ChatOpenAI(model="gpt-4o", temperature=0)
    resp = test_model.invoke([HumanMessage(content="Reply with just: OK")])
    print(f"  ✅ GPT-4o responded: {resp.content.strip()[:50]}")
except Exception as e:
    print(f"  ❌ GPT-4o API FAILED: {str(e)[:150]}")

# ─────────────────────────────────────────────────────────────
# STEP 10: Check disk space
# ─────────────────────────────────────────────────────────────
print("\n💾 Disk space:")
for mount in ["/kaggle/working", "/tmp"]:
    st = os.statvfs(mount)
    free_gb = (st.f_bavail * st.f_frsize) / (1024**3)
    total_gb = (st.f_blocks * st.f_frsize) / (1024**3)
    print(f"  {mount}: {free_gb:.1f} GB free / {total_gb:.1f} GB total")

# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SETUP SUMMARY — MedRAX Premium")
print("=" * 60)
ok_models = sum(1 for v in models_ok.values() if v)
total_models = len(models_ok)
print(f"  Core models:      {ok_models}/{total_models}  {list(k for k,v in models_ok.items() if v)}")
if any(not v for v in models_ok.values()):
    print(f"  Models failed:    {list(k for k,v in models_ok.items() if not v)}")
print(f"  Lazy-load:        LLaVA-Med, MAIRA-2 (downloaded in Cell 2)")
print(f"  GPU:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if torch.cuda.is_available():
    print(f"  VRAM free:        {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")
print(f"  Dataset:          {len(questions)} questions, {total_imgs} images")
print(f"  Model cache:      {MODEL_DIR}")

if ok_models >= 4:
    print(f"\n  🎯 Core models ready — proceed to Cell 2!")
else:
    print(f"\n  ⚠ Only {ok_models}/{total_models} core models — check errors above")

## Cell 2: Load Tools & Agent

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2: LOAD ALL TOOLS + CREATE AGENT WITH CONFLICT RESOLUTION
# ══════════════════════════════════════════════════════════════
import warnings, torch, gc
warnings.filterwarnings("ignore")

from medrax.agent import Agent, ConflictDetector, ConflictResolver
from medrax.tools import *
from medrax.utils import load_prompts_from_file
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = "/tmp/hf_cache"

# ── Load ALL 9 tools ──
# MedRAX Premium tool inventory:
#   1. ImageVisualizerTool       — utility (no model)
#   2. DicomProcessorTool        — utility (no model)
#   3. ChestXRayClassifierTool   — DenseNet (~30MB)
#   4. ChestXRaySegmentationTool — PSPNet (~50MB)
#   5. XRayVQATool               — CheXagent-2-3b (~6GB, bf16)
#   6. ChestXRayReportGeneratorTool — chexpert x2 (~1.5GB)
#   7. LlavaMedTool              — LLaVA-Med-7B (~4GB, 4-bit quantized)
#   8. XRayPhraseGroundingTool   — MAIRA-2 (~7B, 4-bit, gated)
#   9. ChestXRayGeneratorTool    — RoentGen (no public weights)

print("Loading MedRAX Premium tools...\n")
tools = []
tool_status = {}

# ── Group 1: Utility tools (no models) ──
for ToolCls, kwargs, label in [
    (ImageVisualizerTool, {}, "utility"),
    (DicomProcessorTool, {"temp_dir": f"{ROOT}/temp"}, "utility"),
]:
    try:
        tools.append(ToolCls(**kwargs))
        tool_status[ToolCls.__name__] = "✅"
        print(f"  ✓ {ToolCls.__name__} ({label})")
    except Exception as e:
        tool_status[ToolCls.__name__] = f"❌ {str(e)[:60]}"
        print(f"  ✗ {ToolCls.__name__}: {e}")

# ── Group 2: Core model tools (guaranteed to work) ──
for ToolCls, kwargs, label in [
    (ChestXRayClassifierTool, {"device": DEVICE}, "DenseNet ~30MB"),
    (ChestXRaySegmentationTool, {"device": DEVICE}, "PSPNet ~50MB"),
    (ChestXRayReportGeneratorTool, {"cache_dir": MODEL_DIR, "device": DEVICE}, "chexpert ~1.5GB"),
]:
    try:
        tools.append(ToolCls(**kwargs))
        tool_status[ToolCls.__name__] = "✅"
        print(f"  ✓ {ToolCls.__name__} ({label})")
    except Exception as e:
        tool_status[ToolCls.__name__] = f"❌ {str(e)[:60]}"
        print(f"  ✗ {ToolCls.__name__}: {e}")

# ── Group 3: CheXagent VQA (patched, 4-bit to save VRAM for LLaVA-Med) ──
try:
    tools.append(XRayVQATool(cache_dir=MODEL_DIR, device=DEVICE, load_in_4bit=True))
    tool_status["XRayVQATool"] = "✅"
    print(f"  ✓ XRayVQATool (CheXagent-2-3b, 4-bit ~2GB VRAM)")
except Exception as e:
    tool_status["XRayVQATool"] = f"❌ {str(e)[:60]}"
    print(f"  ✗ XRayVQATool: {e}")

gc.collect(); torch.cuda.empty_cache()
if torch.cuda.is_available():
    vram_free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"     VRAM free after core tools: {vram_free:.1f} GB")

# ── Group 4: LLaVA-Med VQA (PUBLIC, 4-bit ~4GB) ──
# Having TWO VQA models creates more conflicts → better conflict resolution demo!
try:
    tools.append(LlavaMedTool(
        model_path="microsoft/llava-med-v1.5-mistral-7b",
        cache_dir=MODEL_DIR,
        device=DEVICE,
        load_in_4bit=True,
        load_in_8bit=False,
    ))
    tool_status["LlavaMedTool"] = "✅"
    print(f"  ✓ LlavaMedTool (LLaVA-Med-7B, 4-bit quantized)")
except Exception as e:
    tool_status["LlavaMedTool"] = f"❌ {str(e)[:80]}"
    print(f"  ✗ LlavaMedTool: {str(e)[:120]}")

gc.collect(); torch.cuda.empty_cache()

# ── Group 5: MAIRA-2 Grounding (GATED — needs HF approval) ──
try:
    tools.append(XRayPhraseGroundingTool(
        model_path="microsoft/maira-2",
        cache_dir=MODEL_DIR,
        device=DEVICE,
        load_in_4bit=True,
    ))
    tool_status["XRayPhraseGroundingTool"] = "✅"
    print(f"  ✓ XRayPhraseGroundingTool (MAIRA-2, 4-bit)")
except Exception as e:
    err = str(e)
    if "gated" in err.lower() or "access" in err.lower() or "401" in err or "403" in err:
        tool_status["XRayPhraseGroundingTool"] = "⚠ GATED"
        print(f"  ✗ XRayPhraseGroundingTool: GATED model — accept terms at huggingface.co/microsoft/maira-2")
    else:
        tool_status["XRayPhraseGroundingTool"] = f"❌ {err[:60]}"
        print(f"  ✗ XRayPhraseGroundingTool: {err[:120]}")

# ── Group 6: RoentGen Generator (no public weights) ──
# RoentGen is a fine-tuned StableDiffusion for CXR generation.
# It requires local weights at /model-weights/roentgen — no public HuggingFace repo exists.
# ChestAgentBench does NOT test generation tasks, so this tool is not needed for benchmarking.
tool_status["ChestXRayGeneratorTool"] = "⚠ No public weights"
print(f"  ─ ChestXRayGeneratorTool: no public HF weights (not needed for ChestAgentBench)")

# ── Summary ──
print(f"\n{'='*60}")
loaded_count = sum(1 for s in tool_status.values() if s == "✅")
print(f"  TOOLS LOADED: {loaded_count}/9")
for name, status in tool_status.items():
    print(f"    {status}  {name}")

if torch.cuda.is_available():
    vram_free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"\n  VRAM free: {vram_free:.1f} GB")

# ── Create Agent with Conflict Resolution ──
prompts = load_prompts_from_file(f"{ROOT}/medrax/docs/system_prompts.txt")
model = ChatOpenAI(model="gpt-4o", temperature=0.2, top_p=0.95)
agent = Agent(
    model,
    tools=tools,
    log_tools=True,
    log_dir=f"{ROOT}/logs",
    system_prompt=prompts["MEDICAL_ASSISTANT"],
    checkpointer=MemorySaver(),
    enable_conflict_resolution=True,
    conflict_sensitivity=0.4,
    deferral_threshold=0.6,
)
print(f"\n✅ Agent ready — conflict resolution ON (sensitivity=0.4, deferral=0.6)")
print(f"   Active tools: {[t.name for t in tools]}")

## Cell 3: Run Benchmark

**Change `NUM_QUESTIONS` below:**
- `25` for a quick test (~30 min)
- `100` for a solid sample (~2 hr)
- `None` for all 2,500 (needs many hours)


In [ ]:
# Pre-run: prepare `selected` set and skip previously-processed questions
# Run this cell first in a new session to compute which questions will run.
import json, random, os

NUM_QUESTIONS = None # change as needed before running this cell (or set to None)
METADATA = f"{ROOT}/chestagentbench/metadata.jsonl"
LOG_DIR = f"{ROOT}/benchmark_logs"
os.makedirs(LOG_DIR, exist_ok=True)
SELECTED_FILE = f"{LOG_DIR}/selected_session.json"
CHECKPOINT_FILE = f"{LOG_DIR}/results_checkpoint.json"

# Load full question set
questions = []
with open(METADATA) as f:
    for line in f:
        questions.append(json.loads(line.strip()))
print(f"Total dataset: {len(questions)} questions")

# Load checkpoint-processed keys if present
processed_keys = set()
if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE) as cf:
            cp = json.load(cf)
        if isinstance(cp, dict) and isinstance(cp.get('all_logs', []), list):
            for entry in cp.get('all_logs', []):
                case = entry.get('case_id')
                qid = entry.get('question_id', '')
                if case:
                    processed_keys.add(f"{case}|{qid}")
            print(f"Checkpoint found: {len(processed_keys)} questions already processed")
        else:
            print("Checkpoint present but invalid format — it will be ignored for selection")
    except Exception as e:
        print(f"Failed to read checkpoint: {e}")

# Build initial selection
if NUM_QUESTIONS:
    random.seed(42)
    selected = random.sample(questions, min(NUM_QUESTIONS, len(questions)))
else:
    selected = questions
print(f"Requested selection size: {len(selected)}")

# Filter processed
if processed_keys:
    filtered = []
    for q in selected:
        key = f"{q.get('case_id')}|{q.get('question_id','')}"
        if key in processed_keys:
            continue
        filtered.append(q)
    selected = filtered
    print(f"After skipping processed: {len(selected)} questions remain")
else:
    print("No processed entries to skip")

# Save selected for Cell 3 to consume
with open(SELECTED_FILE, 'w') as sf:
    json.dump(selected, sf)
print(f"Saved selection to {SELECTED_FILE}")

# Summary: list a few next cases
print('\nNext cases to run (up to 10):')
for q in selected[:10]:
    print('-', q.get('case_id'), q.get('question_id'))


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3: RUN BENCHMARK WITH CONFLICT TRACKING (with checkpoint/resume)
# Robustified: safe globals, precomputed selection support, atomic checkpoint writes
# ══════════════════════════════════════════════════════════════
import json, time, os, base64, random, traceback
from datetime import datetime
from pathlib import Path
from langchain_core.messages import HumanMessage

# Ensure essential globals exist even if cells were run out-of-order
if 'ROOT' not in globals():
    ROOT = os.environ.get('MEDRAX_ROOT', os.getcwd())
if 'MODEL_DIR' not in globals():
    MODEL_DIR = os.environ.get('MODEL_DIR', '/tmp/hf_cache')
LOG_DIR = f"{ROOT}/benchmark_logs"
FIGURES = f"{ROOT}/chestagentbench"
os.makedirs(LOG_DIR, exist_ok=True)

# If a precomputed selection exists (created by the Pre-run cell), load it.
SELECTED_FILE = os.path.join(LOG_DIR, 'selected_session.json')
selected = None
if os.path.exists(SELECTED_FILE):
    try:
        with open(SELECTED_FILE) as sf:
            selected = json.load(sf)
        print(f"Loaded precomputed selected set from {SELECTED_FILE} ({len(selected)} questions)")
    except Exception as e:
        print(f"Failed to load precomputed selection: {e}")
        selected = None

# Legacy fallback: compute selected from METADATA/NUM_QUESTIONS if not precomputed
if selected is None:
    NUM_QUESTIONS = 8   # change to None for all 2500
    METADATA = f"{ROOT}/chestagentbench/metadata.jsonl"
    os.makedirs(LOG_DIR, exist_ok=True)

    # Load questions
    questions = []
    with open(METADATA) as f:
        for line in f:
            questions.append(json.loads(line.strip()))
    print(f"Total dataset: {len(questions)} questions")

    if NUM_QUESTIONS:
        random.seed(42)
        selected = random.sample(questions, min(NUM_QUESTIONS, len(questions)))
    else:
        selected = questions
    print(f"Running on {len(selected)} questions")
else:
    # already ensured LOG_DIR and FIGURES above
    pass

# --------- Enhanced Checkpoint / Resume support ---------
CHECKPOINT_FILE = os.path.join(LOG_DIR, 'results_checkpoint.json')
processed_keys = set()
checkpoint_data = None
checkpoint_valid = False
if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE, "r") as cf:
            cp = json.load(cf)
        # Basic validation: must be dict with required keys
        if isinstance(cp, dict) and 'all_logs' in cp and 'results_by_cat' in cp and isinstance(cp.get('all_logs'), list):
            checkpoint_data = cp
            for entry in cp.get("all_logs", []):
                case = entry.get("case_id")
                qid = entry.get("question_id", "")
                if case:
                    processed_keys.add(f"{case}|{qid}")
            checkpoint_valid = True
            print(f"Resuming from valid checkpoint: {len(processed_keys)} questions already processed")
        else:
            raise ValueError("Checkpoint missing required keys or wrong format")
    except Exception as e:
        # Backup the corrupt checkpoint so it can be inspected later
        ts_bad = datetime.now().strftime("%Y%m%d_%H%M%S")
        bad_path = f"{CHECKPOINT_FILE}.corrupt_{ts_bad}"
        try:
            os.rename(CHECKPOINT_FILE, bad_path)
            print(f"⚠ Corrupt checkpoint moved to: {bad_path} (error: {e})")
        except Exception as e2:
            print(f"⚠ Failed to move corrupt checkpoint: {e2} (original error: {e})")
        checkpoint_valid = False
else:
    print("No checkpoint file found. Starting fresh.")

# If selected was precomputed, it has already excluded processed keys; but if not, filter now
if processed_keys and selected:
    filtered = []
    for q in selected:
        key = f"{q.get('case_id')}|{q.get('question_id','')}"
        if key in processed_keys:
            continue
        filtered.append(q)
    if len(filtered) != len(selected):
        print(f"Filtered out {len(selected)-len(filtered)} already-processed questions from selected")
    selected = filtered
    print(f"Running on {len(selected)} questions (after skipping processed)")
# -----------------------------------------------

# Result trackers
results_by_cat = {k: {"correct": 0, "total": 0} for k in [
    "total", "detection", "classification", "localization", "comparison",
    "relationship", "diagnosis", "characterization", "reasoning"
]}
conflict_log = []
all_logs = []

# If checkpoint data exists and validated, restore rolling state so metrics accumulate across sessions
if checkpoint_valid and checkpoint_data:
    try:
        results_by_cat = checkpoint_data.get('results_by_cat', results_by_cat)
        conflict_log = checkpoint_data.get('conflict_log', conflict_log)
        all_logs = checkpoint_data.get('all_logs', all_logs)
        print(f"Loaded checkpoint: {len(all_logs)} entries, {sum(1 for a in all_logs if 'error' not in a)} successful questions")
    except Exception as e:
        print(f"Warning: failed to restore checkpoint structures: {e}")

print(f"\n{'='*60}")
print(f"  MEDRAX PREMIUM BENCHMARK — {len(selected)} questions")
print(f"{'='*60}\n")

for idx, q in enumerate(selected, 1):
    case_id = q["case_id"]
    cats = q.get("categories", "")
    print(f"[{idx}/{len(selected)}] Case {case_id} | {cats}")

    # Build base64 image content + collect ABSOLUTE file paths for tools
    image_parts = []
    local_paths = []
    for img_path in q.get("images", []):
        full_path = os.path.join(FIGURES, img_path)
        b64 = img_to_base64(full_path)
        if b64:
            image_parts.append({"type": "image_url", "image_url": {"url": b64, "detail": "low"}})
            local_paths.append(full_path)

    if not image_parts:
        print("    ⚠ No images — text-only")

    # Build file-path listing so LLM knows where images are on disk
    if local_paths:
        path_listing = "\n".join(f"  - {p}" for p in local_paths)
        path_block = (
            f"\n\nThe chest X-ray image(s) for this case are available on disk at:\n"
            f"{path_listing}\n"
            f"When calling tools that need an image_path, use these EXACT paths."
        )

    # Capture existing logs BEFORE this question so we can detect new conflict logs
    log_path = Path(f"{ROOT}/logs")
    logs_before = set(log_path.glob("conflict_resolution_*.json")) if log_path.exists() else set()

    try:
        # ...existing code that runs the agent and collects model_answer, correct, is_correct, etc...

        # Safely build the per-question log entry (guard missing variables)
        entry = {"case_id": case_id, "question_id": q.get("question_id", "")}
        entry.update({
            "model_answer": locals().get('model_answer', None),
            "correct_answer": locals().get('correct', None),
            "is_correct": locals().get('is_correct', False),
            "categories": cats,
            "conflicts_occurred": bool(locals().get('q_conflicts', 0) > 0),
            "conflicts_detected": int(locals().get('q_detected', 0)),
            "conflicts_resolved": int(locals().get('q_resolved', 0)),
        })

        all_logs.append(entry)

        # --------- Atomic write checkpoint after each question ---------
        try:
            tmp_cp = CHECKPOINT_FILE + ".tmp"
            with open(tmp_cp, "w") as cf:
                json.dump({"results_by_cat": results_by_cat, "conflict_log": conflict_log, "all_logs": all_logs}, cf, indent=2)
            os.replace(tmp_cp, CHECKPOINT_FILE)
        except Exception as e:
            print(f"⚠️ Failed to write checkpoint atomically: {e}")
        # -----------------------------------------------------

    except Exception as e:
        print(f"  ❌ ERROR: {str(e)[:150]}")
        traceback.print_exc()
        all_logs.append({"case_id": case_id, "error": str(e)[:500], "categories": cats})
        # Don't let one failure poison subsequent questions
        results_by_cat["total"]["total"] += 1
        conflict_log.append({"case_id": case_id, "conflicts_occurred": False, "conflicts_detected": 0, "conflicts_resolved": 0})

# Save final results
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(os.path.join(LOG_DIR, f"results_{ts}.json"), "w") as f:
    json.dump({"results_by_cat": results_by_cat, "conflict_log": conflict_log, "all_logs": all_logs}, f, indent=2)
print(f"\n📝 Saved to {os.path.join(LOG_DIR, f'results_{ts}.json')}")


In [ ]:
# Quick status: show checkpoint + remaining counts
import os, json
cp = None
CHECKPOINT_PATH = os.path.join(LOG_DIR, 'results_checkpoint.json') if 'LOG_DIR' in globals() else None
if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    try:
        with open(CHECKPOINT_PATH) as f:
            cp = json.load(f)
    except Exception as e:
        print(f"Failed to read checkpoint: {e}")

processed = len(cp.get('all_logs', [])) if cp else 0
remaining = len(selected) if 'selected' in globals() and isinstance(selected, list) else 'unknown'
print(f"Checkpoint entries: {processed}")
print(f"Questions to process in this session: {remaining}")
if cp and cp.get('all_logs'):
    last = cp['all_logs'][-1]
    if last:
        print(f"Last processed: case={last.get('case_id')} qid={last.get('question_id')}")
print()

In [ ]:
# Auto-backup: zip checkpoint + results and optionally upload to Kaggle
import os, shutil, subprocess
from datetime import datetime
import glob

# Ensure LOG_DIR exists
if 'LOG_DIR' not in globals():
    LOG_DIR = os.path.join(os.environ.get('MEDRAX_ROOT', os.getcwd()), 'benchmark_logs')
os.makedirs('/kaggle/working', exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_name = f"medrax_benchmark_backup_{ts}"
backup_dir = f"/kaggle/working/{backup_name}"
os.makedirs(backup_dir, exist_ok=True)

# Collect files to include
candidates = set()
# checkpoint and any corrupt checkpoints
if 'CHECKPOINT_FILE' in globals() and os.path.exists(CHECKPOINT_FILE):
    candidates.add(CHECKPOINT_FILE)
# include any .corrupt_* files
if 'CHECKPOINT_FILE' in globals():
    for p in glob.glob(f"{CHECKPOINT_FILE}.corrupt_*"):
        candidates.add(p)
# results and table files
if os.path.exists(LOG_DIR):
    for fname in os.listdir(LOG_DIR):
        if fname.startswith("results_") or fname.startswith("table1_") or fname.endswith('.csv') or fname.endswith('.png'):
            candidates.add(os.path.join(LOG_DIR, fname))

logs_dir = os.path.join(ROOT, "logs") if 'ROOT' in globals() else None
if logs_dir and os.path.exists(logs_dir):
    for fname in os.listdir(logs_dir):
        if fname.startswith("conflict_resolution_") or fname.endswith('.json') or fname.endswith('.log'):
            candidates.add(os.path.join(logs_dir, fname))

# Copy files into backup dir
for f in sorted(candidates):
    try:
        shutil.copy(f, backup_dir)
    except Exception as e:
        print(f"Warning: could not copy {f}: {e}")

# Create zip archive in /kaggle/working
zip_base = os.path.join('/kaggle/working', backup_name)
try:
    shutil.make_archive(zip_base, 'zip', backup_dir)
    zip_path = f"{zip_base}.zip"
    print(f"✅ Backup created: {zip_path}")
except Exception as e:
    print(f"❌ Failed to create zip: {e}")
    zip_path = None

# Optional: attempt to create/upload a Kaggle dataset if CLI + creds present
kaggle_cli = shutil.which('kaggle')
if kaggle_cli and os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
    try:
        print("Attempting to create a Kaggle dataset from the backup directory (may fail if dataset exists)...")
        # First attempt: create dataset
        proc = subprocess.run([
            'kaggle', 'datasets', 'create', '-p', backup_dir, '-m', f'MedRAX benchmark backup {ts}'
        ], capture_output=True, text=True, timeout=300)
        print(proc.stdout)
        if proc.returncode != 0:
            print('Create failed, trying `datasets version` to upload as a new version (if dataset exists)...')
            proc2 = subprocess.run([
                'kaggle', 'datasets', 'version', '-p', backup_dir, '-m', f'backup {ts}'
            ], capture_output=True, text=True, timeout=300)
            print(proc2.stdout)
    except Exception as e:
        print(f"Kaggle upload attempt failed: {e}")
else:
    print("Kaggle CLI or credentials not found — backup is saved in /kaggle/working for manual download.")

print('\nFiles included in backup:')
for f in sorted(candidates):
    print(' -', f)


## Cell 4: Results — Conflict Metrics & Table 1

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4: CONFLICT METRICS + TABLE 1 COMPARISON
# ══════════════════════════════════════════════════════════════
import pandas as pd

processed = [x for x in all_logs if "error" not in x]
total_q = len(processed)
# conflicts_occurred is now True/False per question
questions_with_conflicts = sum(1 for c in conflict_log if c.get("conflicts_occurred"))
total_detected  = sum(c["conflicts_detected"] for c in conflict_log)
total_resolved  = sum(c["conflicts_resolved"] for c in conflict_log)

print("=" * 60)
print("  CONFLICT RESOLUTION METRICS")
print("=" * 60)
print(f"  Questions processed:          {total_q}")
print(f"  Questions with conflicts:     {questions_with_conflicts}")
print(f"  Total conflicts detected:     {total_detected}")
print(f"  Total conflicts resolved:     {total_resolved}")
print()

occ_rate  = (questions_with_conflicts / total_q * 100) if total_q > 0 else 0
det_ratio = (total_detected / total_q * 100) if total_q > 0 else 0  # conflicts per question
res_ratio = (total_resolved / total_detected * 100) if total_detected > 0 else 0

print(f"  Conflict Occurrence Rate:     {occ_rate:.1f}%  ({questions_with_conflicts}/{total_q} questions had conflicts)")
print(f"  Avg Conflicts per Question:   {total_detected/total_q:.2f}" if total_q > 0 else "  N/A")
print(f"  Conflict Resolve Ratio:       {res_ratio:.1f}%  ({total_resolved}/{total_detected} detected conflicts resolved)")

# ── TABLE 1 ──
print("\n" + "=" * 60)
print("  TABLE 1: ChestAgentBench Accuracy (%)")
print("=" * 60)

categories = ['Detection','Classification','Localization','Comparison',
              'Relationship','Diagnosis','Characterization','Overall']
cat_keys   = ['detection','classification','localization','comparison',
              'relationship','diagnosis','characterization','total']

paper = {
    'LLaVA-Med':       [32.4, 30.8, 30.2, 30.6, 31.8, 29.3, 28.8, 28.7],
    'CheXagent':       [38.7, 34.7, 42.5, 38.5, 39.8, 33.5, 34.2, 39.5],
    'Llama-3.2-90B':   [58.1, 56.5, 59.9, 57.5, 59.3, 55.9, 58.0, 57.9],
    'GPT-4o':          [58.7, 54.6, 59.0, 55.5, 59.0, 52.6, 56.1, 56.4],
    'MedRAX':          [64.1, 62.9, 63.6, 61.8, 63.1, 62.5, 64.0, 63.1],
}

ours = []
for k in cat_keys:
    d = results_by_cat[k]
    ours.append(round(d["correct"] / d["total"] * 100, 1) if d["total"] > 0 else 0.0)
paper['MedRAX_Premium'] = ours

df = pd.DataFrame(paper, index=categories)
print(df.to_string())

csv_path = f"{LOG_DIR}/table1_{ts}.csv"
df.to_csv(csv_path)
print(f"\n📝 Table saved: {csv_path}")

## Cell 5: Charts

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5: VISUALIZATION
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ── LEFT: Bar chart — Table 1 comparison ──
ax = axes[0]
cats_plot = ['Detection','Classification','Localization','Comparison',
             'Relationship','Diagnosis','Characterization','Overall']
x = np.arange(len(cats_plot))
width = 0.13
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99','#ff99cc','#c299ff']
models = list(paper.keys())

for i, (m, c) in enumerate(zip(models, colors)):
    ax.bar(x + i * width, paper[m], width, label=m, color=c, edgecolor='black', linewidth=0.5)

ax.set_ylabel('Accuracy (%)')
ax.set_title('Table 1: ChestAgentBench Performance', fontweight='bold')
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(cats_plot, rotation=45, ha='right')
ax.legend(fontsize=7, loc='upper left')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

# ── RIGHT: Conflict metrics pie/bar ──
ax2 = axes[1]
if total_detected > 0:
    labels = ['Resolved', 'Unresolved']
    sizes = [total_resolved, total_detected - total_resolved]
    colors_pie = ['#2ecc71', '#e74c3c']
    ax2.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
    ax2.set_title(f'Conflict Resolution\n({total_detected} conflicts from {total_q} questions)', fontweight='bold')
else:
    bars = ax2.bar(['Occurrence\nRate', 'Detection\nRatio', 'Resolve\nRatio'],
                   [occ_rate, det_ratio, res_ratio], color=['#3498db','#2ecc71','#e67e22'])
    ax2.set_ylabel('%')
    ax2.set_ylim(0, 100)
    ax2.set_title('Conflict Metrics', fontweight='bold')
    for b in bars:
        ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
                f'{b.get_height():.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(f"{LOG_DIR}/results_chart.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📝 Chart saved: {LOG_DIR}/results_chart.png")